# TirraMind Phase 50 — HetTGN Retrain with Price Features + Residual Returns

**Phase 50 changes (all in code — auto-cloned from GitHub):**
- `graph_builder.py`: 9-dim price-derived instrument features (momentum, vol, volume, drawdown, sharpe)
- `trainer.py`: cross-sectionally demeaned residual return targets
- `het_tgn.py`: return_pred_head 2→3 layers (more capacity)

**Training config:** hidden=128, heads=4, residual returns, ListNet + auto-tune, return_weight=3.0

## Before Running
1. Attach `tirramind-data` dataset (pipeline.db)
2. Set `tirramind_token` Kaggle Secret (GitHub PAT with read access)
3. Accelerator: GPU T4 x1

In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

pip("torch-geometric==2.7.0")
import torch
torch_ver = torch.__version__.split("+")[0]
cuda_runtime = torch.version.cuda
cuda_tag = f"cu{cuda_runtime.replace('.', '')}" if cuda_runtime else "cpu"
wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
print(f"Installing PyG extras for torch={torch_ver}, runtime={cuda_tag}")
pip("torch-scatter", "torch-sparse", "-f", wheel_url)
pip("tqdm", "rich", "wandb")
print("All dependencies installed.")

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

# Clone latest code from GitHub
from kaggle_secrets import UserSecretsClient
_token = UserSecretsClient().get_secret("tirramind_token")
_repo_url = f"https://{_token}@github.com/savabs/tirramind.git"

subprocess.run(
    ["git", "clone", "--depth=1", _repo_url, str(WORK_DIR)],
    check=True, capture_output=True
)
print(f"Cloned latest code → {WORK_DIR}")

# Copy pipeline.db from dataset
def find_data_root(root="/kaggle/input"):
    for dirpath, dirs, files in os.walk(root):
        if "pipeline.db" in set(files):
            return Path(dirpath)
    return None

data_root = find_data_root()
assert data_root is not None, "Attach tirramind-data dataset"

pipeline_dir = WORK_DIR / ".tirra_pipeline"
pipeline_dir.mkdir(exist_ok=True)
shutil.copy2(data_root / "pipeline.db", pipeline_dir / "pipeline.db")
print(f"Copied pipeline.db ({(pipeline_dir / 'pipeline.db').stat().st_size // 1_000_000} MB)")

# Patch pipeline __init__ for lazy import
pipeline_init = WORK_DIR / "agent" / "pipeline" / "__init__.py"
pipeline_init.write_text(
    '"""TirraMind — Pipeline Layer."""\n\n'
    'from agent.pipeline.storage_backend import (\n'
    '    PostgresBackend,\n'
    '    SQLiteBackend,\n'
    '    StorageBackend,\n'
    ')\n'
    'from agent.pipeline.store import PipelineStore\n\n'
    '__all__ = ["PipelineStore", "PipelineScheduler", "StorageBackend", "SQLiteBackend", "PostgresBackend"]\n\n\n'
    'def __getattr__(name: str):\n'
    '    if name == "PipelineScheduler":\n'
    '        from agent.pipeline.scheduler import PipelineScheduler\n'
    '        return PipelineScheduler\n'
    '    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")\n',
    encoding="utf-8"
)
print("Setup complete.")

In [ ]:
import torch
import torch_geometric
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, PyG: {torch_geometric.__version__}")
print(f"Device: {DEVICE}")

In [ ]:
from pathlib import Path
WORK_DIR = Path("/kaggle/working/tirramind_v1")
assert (WORK_DIR / ".tirra_pipeline/pipeline.db").exists()
for f in ["agent/models/gnn/graph_builder.py", "agent/models/gnn/het_tgn.py", "agent/models/gnn/trainer.py", "scripts/retrain_gnn.py"]:
    assert (WORK_DIR / f).exists(), f"Missing: {f}"
print("All checks passed. Starting Phase 50 training from epoch 0.")

In [ ]:
import subprocess
import sys
import os
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
CKPT_DIR = WORK_DIR / ".tirra_pipeline" / "checkpoints" / "phase50"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = globals().get("DEVICE", "cpu")

# W&B setup
_wandb_project = None
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = _wandb_key
    _wandb_project = "tirramind"
    print("W&B enabled")
except Exception as _e:
    print(f"W&B disabled ({_e.__class__.__name__})")

print(f"Device: {DEVICE}")
print("Phase 50: hidden=128, heads=4, price features, residual returns, ListNet + auto-tune")

cmd = [
    sys.executable, "scripts/retrain_gnn.py",
    "--epochs",              "30",
    "--hidden-dim",          "128",
    "--num-layers",          "2",
    "--num-heads",           "4",
    "--lr",                  "1e-3",
    "--backup",
    "--window-size",         "604800",
    "--gdelt-frac",          "0.05",
    "--max-windows",         "200",
    "--auto-tune",
    "--listnet",
    "--return-weight",       "3.0",
    "--return-log-var-max",  "0.0",
    "--direction-loss",
    "--residual-returns",
    "--device",              DEVICE,
    "--skip-eval",
    "--checkpoint-dir",      str(CKPT_DIR),
    "--model-out",           ".tirra_pipeline/gnn_model_phase50.pt",
]

if _wandb_project:
    cmd += [
        "--wandb-project", _wandb_project,
        "--wandb-run",     "phase50-ep1-30",
        "--wandb-tags",    "phase50,price-features,residual-returns",
    ]

print("Running:", " ".join(cmd))
print("-" * 70)

process = subprocess.Popen(
    cmd, cwd=str(WORK_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed: exit {return_code}")
print("\nPhase 50 training completed.")

In [ ]:
import shutil
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
CKPT_DIR = WORK_DIR / ".tirra_pipeline" / "checkpoints" / "phase50"
OUT_DIR  = Path("/kaggle/working")

# Copy final model
final_model = WORK_DIR / ".tirra_pipeline" / "gnn_model_phase50.pt"
if final_model.exists():
    shutil.copy2(final_model, OUT_DIR / "gnn_model_phase50.pt")
    print(f"gnn_model_phase50.pt → /kaggle/working/ ({final_model.stat().st_size / 1_000_000:.1f} MB)")

# Copy checkpoints
for ckpt in sorted(CKPT_DIR.glob("epoch_*.pt")):
    shutil.copy2(ckpt, OUT_DIR / ckpt.name)
    print(f"{ckpt.name} → /kaggle/working/")

print("\nDownload all files from the Output tab.")

In [ ]:
# Optional: run backtest directly on Kaggle to see IC before downloading
import subprocess, sys, shutil
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
MODEL = WORK_DIR / ".tirra_pipeline" / "gnn_model_phase50.pt"
assert MODEL.exists(), "Model not found — training may not have completed"

# Symlink to expected name
link = WORK_DIR / ".tirra_pipeline" / "gnn_model.pt"
if link.exists():
    link.unlink()
shutil.copy2(MODEL, link)

result = subprocess.run(
    [sys.executable, str(WORK_DIR / "scripts/phase40_gnn_backtest.py"), "--out", ".tirra_pipeline/ic_results_phase50.json"],
    cwd=str(WORK_DIR), text=True
)

if result.returncode == 0:
    print("\nBacktest complete. Check output above for IC results.")
else:
    print("Backtest FAILED")